In [20]:
import numpy as np, pandas as pd, random, os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow import keras

In [2]:
df = pd.read_csv("../data/TP_RNN_meteo.csv")

In [3]:
TARGET = "T (degC)"

In [4]:
target_idx = df.columns.get_loc(TARGET)

In [5]:
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)
train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

In [6]:
from sklearn.preprocessing import StandardScaler

In [7]:
scaler = StandardScaler().fit(train_df)
train_s = scaler.transform(train_df)
val_s = scaler.transform(val_df)
test_s = scaler.transform(test_df)

In [8]:
def make_windows(data, target_idx, past=24, horizon=1):
    X, y = [], []
    for i in range(len(data) - past - horizon + 1):
        X.append(data[i : i + past])                       # girdi: t-24 ... t-1
        y.append(data[i + past + horizon - 1, target_idx]) # hedef: t+horizon-1
    return np.array(X), np.array(y)

In [9]:
X_train, y_train = make_windows(train_s, target_idx)
X_val, y_val = make_windows(val_s, target_idx)
X_test, y_test = make_windows(test_s, target_idx)

In [24]:
mean_T, std_T = scaler.mean_[target_idx], scaler.scale_[target_idx]
to_c = lambda z: z * std_T + mean_T
true_test_c = to_c(y_test)

In [10]:
print(X_train.shape, y_train.shape)

(49039, 24, 19) (49039,)


In [11]:
mean_T, std_T = scaler.mean_[target_idx], scaler.scale_[target_idx]
persist_pred = X_test[:, -1, target_idx] * std_T + mean_T   # son bilinen T
true_T = y_test * std_T + mean_T

In [12]:
print("Test persistence MAE:", np.mean(np.abs(persist_pred - true_T)))

Test persistence MAE: 0.6761963775023832


In [13]:
def report(name, pred_c):
    mae = np.mean(np.abs(pred_c - true_test_c))
    rmse = np.sqrt(np.mean((pred_c - true_test_c) ** 2))
    print(f"{name:<14} MAE={mae:.3f} °C   RMSE={rmse:.3f} °C")

In [14]:
def build_model(cell, units=64):
    layer = {"simple": keras.layers.SimpleRNN,
             "lstm": keras.layers.LSTM,
             "gru": keras.layers.GRU}[cell]
    model = keras.Sequential([
        keras.layers.Input(shape=X_train.shape[1:]),
        layer(units),
        keras.layers.Dense(1),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
    return model

In [16]:
import tensorflow as tf

In [21]:
SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

In [25]:
results = {}
for cell in ["simple", "lstm", "gru"]:
    tf.random.set_seed(SEED)
    model = build_model(cell)
    early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    hist = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                     epochs=30, batch_size=128, callbacks=[early], verbose=0)
    pred = model.predict(X_test, verbose=0).ravel()
    report(cell.upper(), to_c(pred))
    print(f"   epoch={len(hist.history['loss'])}  best val_loss={min(hist.history['val_loss']):.5f}")
    results[cell] = model
    model.save("../model/automatants_model.keras")
report("PERSISTENCE", to_c(X_test[:, -1, target_idx]))

SIMPLE         MAE=0.471 °C   RMSE=0.664 °C
   epoch=19  best val_loss=0.00661
LSTM           MAE=0.441 °C   RMSE=0.629 °C
   epoch=25  best val_loss=0.00593
GRU            MAE=0.442 °C   RMSE=0.627 °C
   epoch=30  best val_loss=0.00590
PERSISTENCE    MAE=0.676 °C   RMSE=0.960 °C
